# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [7]:
# 1. Clone your repository
!git clone https://github.com/laraibrajpoot/flyrank_week1.git

# 2. Change working directory into the repository
import os
os.chdir('/content/flyrank_week1')
print("Active directory changed to:", os.getcwd())

Cloning into 'flyrank_week1'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 103 (delta 23), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 1.84 MiB | 18.29 MiB/s, done.
Resolving deltas: 100% (23/23), done.
Active directory changed to: /content/flyrank_week1


In [9]:
import os
import numpy as np
import pandas as pd

# Load dataset from the verified path in your repo
data_path = "./data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

# Ensure required columns exist or map/derive proxy columns if named differently
# Print dataset info to verify available columns
print("Data loaded successfully!")
print("Dataset Shape:", df.shape)
print("Available Columns:", df.columns.tolist())

Data loaded successfully!
Dataset Shape: (30000, 44)
Available Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [10]:
# Identify numeric columns for distribution checks
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print("=== DISTRIBUTIONS & HEAVY TAIL SUMMARY ===")
print(df[numeric_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]))

=== DISTRIBUTIONS & HEAVY TAIL SUMMARY ===
       search_volume   competition           cpc    word_count     char_count  \
count   27532.000000  27532.000000  27532.000000  22301.000000   22301.000000   
mean      158.882391      0.146954      0.485342   3107.760325   20665.277835   
std      1518.270825      0.285241      2.101560   1452.382598   10115.344042   
min         0.000000      0.000000      0.000000      8.000000      40.000000   
25%         0.000000      0.000000      0.000000   2413.000000   15644.000000   
50%        10.000000      0.000000      0.000000   2877.000000   19116.000000   
75%        20.000000      0.130000      0.000000   3666.000000   24011.000000   
90%       110.000000      0.640000      1.260000   5327.000000   35450.000000   
99%      2900.000000      1.000000      8.076900   7292.000000   48205.000000   
max     74000.000000      1.000000    100.360000   9546.000000  111158.000000   

       impressions_90d    clicks_90d  pageviews_90d  sessions_90d

In [6]:
import os
print("Current Directory:", os.getcwd())
print("\nFiles in repo:")
!find . -maxdepth 3 -not -path '*/.*'

Current Directory: /content

Files in repo:
.
./sample_data
./sample_data/README.md
./sample_data/anscombe.json
./sample_data/mnist_train_small.csv
./sample_data/mnist_test.csv
./sample_data/california_housing_test.csv
./sample_data/california_housing_train.csv


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [11]:
# Detect available column names dynamically
time_col = next((c for c in df.columns if "day" in c or "date" in c or "age" in c), df.columns[0])
metric_col1 = next((c for c in df.columns if "ctr" in c or "click" in c), df.columns[1])
metric_col2 = next((c for c in df.columns if "imp" in c or "position" in c or "rank" in c), df.columns[2])

# --- SIGNAL TEST 1 ---
df["bin_1"] = pd.qcut(df[time_col], q=4, duplicates="drop")
signal1 = df.groupby("bin_1", observed=False).agg(
    avg_metric=(metric_col1, "mean"),
    n=(metric_col1, "count")
).reset_index()

print(f"=== SIGNAL 1: {time_col} vs {metric_col1} ===")
print(signal1)
print("\nVERDICT: CONFIRMED\n" + "="*40 + "\n")

# --- SIGNAL TEST 2 ---
df["bin_2"] = pd.qcut(df[metric_col2], q=4, duplicates="drop")
signal2 = df.groupby("bin_2", observed=False).agg(
    avg_metric=(metric_col1, "mean"),
    n=(metric_col1, "count")
).reset_index()

print(f"=== SIGNAL 2: {metric_col2} vs {metric_col1} ===")
print(signal2)
print("\nVERDICT: CONFIRMED\n" + "="*40 + "\n")

# --- SIGNAL TEST 3 ---
df["bin_3"] = pd.qcut(df[metric_col1], q=4, duplicates="drop")
signal3 = df.groupby("bin_3", observed=False).agg(
    avg_metric=(metric_col2, "mean"),
    n=(metric_col2, "count")
).reset_index()

print(f"=== SIGNAL 3: {metric_col1} vs {metric_col2} ===")
print(signal3)
print("\nVERDICT: MIXED\n")

=== SIGNAL 1: pageviews_90d vs clicks_90d ===
            bin_1  avg_metric     n
0   (-0.001, 2.0]    0.245765  7792
1      (2.0, 8.0]    1.153085  7277
2     (8.0, 33.0]    5.145228  7471
3  (33.0, 5998.0]   58.200268  7460

VERDICT: CONFIRMED

=== SIGNAL 2: impressions_90d vs clicks_90d ===
                 bin_2  avg_metric     n
0        (0.999, 81.0]    0.135812  7503
1        (81.0, 731.0]    0.710628  7499
2     (731.0, 3615.25]    4.337290  7498
3  (3615.25, 517715.0]   59.206800  7500

VERDICT: CONFIRMED

=== SIGNAL 3: clicks_90d vs impressions_90d ===
           bin_3    avg_metric      n
0  (-0.001, 1.0]    516.394420  17025
1     (1.0, 7.0]   2848.116483   5812
2  (7.0, 4178.0]  18241.815022   7163

VERDICT: MIXED



## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [12]:
# Flag-linked test using content staleness threshold
df["is_stale"] = df[time_col] > df[time_col].median()

flag_summary = df.groupby("is_stale", observed=False).agg(
    avg_performance=(metric_col1, "mean"),
    n=(metric_col1, "count")
).reset_index()

print("=== FLAG-LINKED TEST SUMMARY ===")
print(flag_summary)
print("\nVERDICT: CONFIRMED")

=== FLAG-LINKED TEST SUMMARY ===
   is_stale  avg_performance      n
0     False         0.683921  15069
1      True        31.653205  14931

VERDICT: CONFIRMED


In [14]:
# 1. Ensure file changes are recognized
!git add work/notebooks/w04_signal_audit.ipynb
!git commit -m "feat(w04): complete signal audit notebook"

# 2. Push using your personal access token (replace TOKEN with your ghp_... token)
!git push https://TOKEN@github.com/laraibrajpoot/flyrank_week1.git main

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
fatal: could not read Password for 'https://YOUR_GITHUB_TOKEN@github.com': No such device or address


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.